In [11]:
import os
import csv
import time
import json
import math
import inspect
import itertools
from dataclasses import dataclass, asdict
import numpy as np
import gc

import torch
import torch.nn as nn
from torch.nn import functional as F

# --- CENTRAL CONFIGURATION ---
@dataclass
class ExperimentConfig:
    run_name: str = "baseline"
    seed: int = 1337
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    
    # Standard Hyperparameters (Matching original nanoGPT train_shakespeare_char.py)
    batch_size: int = 64
    block_size: int = 256
    max_iters: int = 5000
    eval_interval: int = 250
    eval_iters: int = 200
    learning_rate: float = 1e-3
    min_lr: float = 1e-4        # Used for cosine decay
    warmup_iters: int = 100     # Used for cosine decay
    weight_decay: float = 1e-1
    grad_clip: float = 1.0
    vocab_size: int = 65

    log_interval: int = 10 

    # Model Architecture
    n_layer: int = 6
    n_head: int = 6
    n_embd: int = 384
    dropout: float = 0.2
    
    # --- ABLATION FLAGS ---
    norm_type: str = "layernorm"   # Options: 'layernorm', 'rmsnorm, 'none'
    norm_placement: str = "pre"    # Options: 'pre', 'post'
    linear_type: str = "standard"  # Options: 'standard'
    pos_encoding: str = "learned"         # 'learned', 'rope', 'alibi'
    residual_type: str = "standard"# Options: 'standard', 'scaled', 'none'
    activation_type: str = "gelu"

#similar to original code
'''what was removed

Learning Rate Decay (Cosine Schedule): The original code doesn't just use a flat 1e-3 learning rate. It starts at 0 (warmup), goes up to 1e-3, and then slowly curves down to 1e-4 like a cosine wave.
Weight Decay Splitting: It applies weight_decay=0.1 to matrices, but weight_decay=0.0 to biases and LayerNorms. This version just applies standard AdamW to everything equally.
Gradient Clipping: The original code prevents exploding math by clipping gradients (grad_clip = 1.0). This code lets the gradients flow naturally.'''

"what was removed\n\nLearning Rate Decay (Cosine Schedule): The original code doesn't just use a flat 1e-3 learning rate. It starts at 0 (warmup), goes up to 1e-3, and then slowly curves down to 1e-4 like a cosine wave.\nWeight Decay Splitting: It applies weight_decay=0.1 to matrices, but weight_decay=0.0 to biases and LayerNorms. This version just applies standard AdamW to everything equally.\nGradient Clipping: The original code prevents exploding math by clipping gradients (grad_clip = 1.0). This code lets the gradients flow naturally."

In [67]:
#THIS IS WHERE NEW ARCHITECTURES GO ***HEREEEEE***
# 1. Modular Normalization Builder
def get_norm_layer(config, ndim):
    if config.norm_type == "layernorm":
        return nn.LayerNorm(ndim, bias=False)
    elif config.norm_type == "rmsnorm":
        return nn.RMSNorm(ndim)
    elif config.norm_type == "none":
        return nn.Identity() #does absolutely nothing
    else:
        raise ValueError(f"Unknown norm_type: {config.norm_type}")

# 2. Modular Linear Builder
def get_linear_layer(config, in_features, out_features, bias=False):
    if config.linear_type == "standard":
        return nn.Linear(in_features, out_features, bias=bias)
    # elif config.linear_type == "bitlinear": # QUANTISATION WILL BE ADDED LATER
    #     return BitLinear(in_features, out_features)
    else:
        raise ValueError(f"Unknown linear_type: {config.linear_type}")

In [13]:
# helper functions for RoPE & ALiBi
def precompute_freqs_cis(dim: int, end: int, theta: float = 10000.0):
    """
    Precompute the frequency complex numbers for RoPE.
    dim: head dimension (n_embd // n_head)
    end: max sequence length (block_size)
    """
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))
    t = torch.arange(end, device=freqs.device)
    freqs = torch.outer(t, freqs).float()  # (end, dim//2)
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs)  # complex64
    return freqs_cis

def rotate_half(x):
    """Rotates half the hidden dimensions."""
    x1, x2 = x[..., : x.shape[-1] // 2], x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_emb(xq, xk, freqs_cis):
    # xq, xk: (B, H, T, D)
    # freqs_cis: (T, D//2)
    B, H, T, D = xq.shape
    # reshape to (B, H, T, D//2, 2)
    xq_ = xq.float().reshape(B, H, T, D//2, 2)
    xk_ = xk.float().reshape(B, H, T, D//2, 2)
    # view as complex
    xq_complex = torch.view_as_complex(xq_)
    xk_complex = torch.view_as_complex(xk_)
    # expand freqs_cis to (1, 1, T, D//2)
    freqs_cis = freqs_cis.unsqueeze(0).unsqueeze(0)  # (1, 1, T, D//2)
    # multiply
    xq_out = torch.view_as_real(xq_complex * freqs_cis).flatten(3)
    xk_out = torch.view_as_real(xk_complex * freqs_cis).flatten(3)
    return xq_out.type_as(xq), xk_out.type_as(xk)

In [14]:
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        
        self.c_attn = get_linear_layer(config, config.n_embd, 3 * config.n_embd)
        self.c_proj = get_linear_layer(config, config.n_embd, config.n_embd)
        
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.dropout = config.dropout
        self.head_dim = config.n_embd // config.n_head
        
        self.pos_encoding = config.pos_encoding
        
        # For RoPE: precompute frequency cis
        if self.pos_encoding == 'rope':
            self.register_buffer("freqs_cis", precompute_freqs_cis(self.head_dim, config.block_size))
        
        # For ALiBi: precompute slopes
        if self.pos_encoding == 'alibi':
            # ALiBi slopes: 2^{-8/i} or 2^{-(2i+1)/8}? Standard: 2^{-(i+1)*8/n_head}
            # Following original paper: slopes = 2^{-(i+1)*8/n_head}
            slopes = torch.tensor([2 ** (-(i + 1) * 8.0 / config.n_head) for i in range(config.n_head)])
            self.register_buffer("alibi_slopes", slopes.view(1, config.n_head, 1, 1))
        
        # Causal mask (for manual attention, used by both)
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                     .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)  # (B, nh, T, hd)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        
        # ----- RoPE: rotate q and k -----
        if self.pos_encoding == 'rope':
            freqs_cis = self.freqs_cis[:T]  # (T, hd//2)
            q, k = apply_rotary_emb(q, k, freqs_cis)
        
        # ----- Attention scores -----
        # Flash Attention (Massive speedup and VRAM reduction)
        if hasattr(torch.nn.functional, 'scaled_dot_product_attention'):
            y = torch.nn.functional.scaled_dot_product_attention(
                q, k, v, attn_mask=None, dropout_p=self.dropout if self.training else 0, is_causal=True
            )
        else:
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        
        # Apply causal mask (triangular)
            att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
        
        # ----- ALiBi: add distance-based bias -----
        if self.pos_encoding == 'alibi':
            # bias shape: (1, nh, T, T), distance = j - i
            # Each head gets its own slope: slope * (j - i)
            position_ids = torch.arange(T, device=x.device)
            distance = position_ids[None, :] - position_ids[:, None]  # (T, T)
            # Only apply to positions j >= i (causal), but we can add full then mask will set -inf to upper
            alibi_bias = self.alibi_slopes * distance.unsqueeze(0)  # (1, nh, T, T)
            att = att + alibi_bias
        
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v
        
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y
        
class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        if config.activation_type == "swiglu":
            hidden_dim = int((8/3)*config.n_embd)
            self.w1 = get_linear_layer(config, config.n_embd, hidden_dim)
            self.w2 = get_linear_layer(config, config.n_embd, hidden_dim)
            self.w3 = get_linear_layer(config, hidden_dim, config.n_embd)
            self.silu = nn.SiLU()

        else:
            self.c_fc   = get_linear_layer(config, config.n_embd, 4 * config.n_embd)
            self.gelu   = nn.GELU()
            self.c_proj = get_linear_layer(config, 4 * config.n_embd, config.n_embd)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        if self.config.activation_type == "swiglu":
            gate = self.silu(self.w1(x))
            data = self.w1(x)
            x = self.w3(gate*data)
        else:
            x = self.c_fc(x)
            x = self.gelu(x)
            x = self.c_proj(x)
        return self.dropout(x)

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        # Uses our modular norm builder!
        self.config = config
        self.ln_1 = get_norm_layer(config, config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = get_norm_layer(config, config.n_embd)
        self.mlp = MLP(config)

        if self.config.residual_type == "scaled":
            self.res_scale = 1/ math.sqrt(self.config.n_layer)
        else:
            self.res_scale = 1

    def forward(self, x):
        if self.config.norm_placement == "post":
            if self.config.residual_type == "none":
                x = self.ln_1(self.attn(x))
                x = self.ln_2(self.mlp(x))
            else:
                x = self.ln_1(x + self.res_scale*self.attn(x))
                x = self.ln_2(x + self.res_scale*self.mlp(x))
        else:
            if self.config.residual_type == "none":
                x = self.attn(self.ln_1(x))
                x = self.mlp(self.ln_2(x))
            else:
                x = x + self.res_scale*self.attn(self.ln_1(x))
                x = x + self.res_scale*self.mlp(self.ln_2(x))
        return x

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.wte = nn.Embedding(config.vocab_size, config.n_embd)
        
        # Only learned positional embeddings are used as a separate embedding
        self.wpe = nn.Embedding(config.block_size, config.n_embd) if config.pos_encoding == 'learned' else None
        
        self.drop = nn.Dropout(config.dropout)
        self.h = nn.ModuleList([Block(config) for _ in range(config.n_layer)])
        self.ln_f = get_norm_layer(config, config.n_embd)
        self.lm_head = get_linear_layer(config, config.n_embd, config.vocab_size)
        
        # weight tying
        self.wte.weight = self.lm_head.weight
        
        self.apply(self._init_weights)
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02/math.sqrt(2 * config.n_layer))
                
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if getattr(module, 'bias', None) is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def configure_optimizers(self, weight_decay, learning_rate, betas, device_type):
        """Splits weights so only 2D matrices get weight decay (Like the original nanoGPT)"""
        param_dict = {pn: p for pn, p in self.named_parameters() if p.requires_grad}
        decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
        nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]
        optim_groups = [
            {'params': decay_params, 'weight_decay': weight_decay},
            {'params': nodecay_params, 'weight_decay': 0.0}
        ]
        use_fused = 'fused' in inspect.signature(torch.optim.AdamW).parameters and device_type == 'cuda'
        extra_args = dict(fused=True) if use_fused else dict()
        return torch.optim.AdamW(optim_groups, lr=learning_rate, betas=betas, **extra_args)

    def forward(self, idx, targets=None):
        device = idx.device
        b, t = idx.size()
        tok_emb = self.wte(idx)
        
        if self.wpe is not None:
            # learned absolute positional embeddings
            pos = torch.arange(0, t, dtype=torch.long, device=device)
            pos_emb = self.wpe(pos)
            x = self.drop(tok_emb + pos_emb)
        else:
            # RoPE or ALiBi: no addition here
            x = self.drop(tok_emb)
        
        for block in self.h:
            x = block(x)
        x = self.ln_f(x)

        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        else:
            logits = self.lm_head(x[:, [-1], :])
            loss = None
        return logits, loss

In [69]:
# --- LOGGING HELPER ---
def log_metrics_to_csv(config, metrics, filepath="output/metrics.csv"):
    row_data = {**asdict(config), **metrics}
    file_exists = os.path.isfile(filepath)
    with open(filepath, mode='a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=row_data.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(row_data)

# --- MAIN TRAINING LOOP ---
def train_run(config: ExperimentConfig):
    print(f"\n{'='*50}\nStarting Run: {config.run_name}\n{'='*50}")
    
    torch.manual_seed(config.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(config.seed)

    # Autocast setup
    device_type = 'cuda' if 'cuda' in config.device else 'cpu'
    ptdtype = torch.float16 if device_type == 'cuda' else torch.bfloat16

    # Data Loading
    train_data = np.memmap('data/train.bin', dtype=np.uint16, mode='r')
    val_data = np.memmap('data/val.bin', dtype=np.uint16, mode='r')

    def get_batch(split):
        data = train_data if split == 'train' else val_data
        ix = torch.randint(len(data) - config.block_size, (config.batch_size,))
        x = torch.stack([torch.from_numpy((data[i:i+config.block_size]).astype(np.int64)) for i in ix])
        y = torch.stack([torch.from_numpy((data[i+1:i+1+config.block_size]).astype(np.int64)) for i in ix])
        if device_type == 'cuda':
            # Async transfer to GPU (massive speedup)
            x, y = x.pin_memory().to(config.device, non_blocking=True), y.pin_memory().to(config.device, non_blocking=True)
        else:
            x, y = x.to(config.device), y.to(config.device)
        return x, y

    # Model Setup
    model = GPT(config).to(config.device)
    optimizer = model.configure_optimizers(config.weight_decay, config.learning_rate, (0.9, 0.95), device_type)
    scaler = torch.amp.GradScaler(device_type, enabled=(device_type == 'cuda'))

    @torch.no_grad()
    def estimate_loss():
        out = {}
        model.eval()
        for split in ['train', 'val']:
            losses = torch.zeros(config.eval_iters)
            for k in range(config.eval_iters):
                X, Y = get_batch(split)
                with torch.autocast(device_type=device_type, dtype=ptdtype):
                    _, loss = model(X, Y)
                losses[k] = loss.item()
            out[split] = losses.mean().item()
        model.train()
        return out

    # Cosine Learning Rate Schedule
    def get_lr(it):
        if it < config.warmup_iters:
            return config.learning_rate * (it + 1) / (config.warmup_iters + 1)
        if it > config.max_iters:
            return config.min_lr
        decay_ratio = (it - config.warmup_iters) / (config.max_iters - config.warmup_iters)
        coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
        return config.min_lr + coeff * (config.learning_rate - config.min_lr)

    # Training Loop
    start_time = time.time()
    loss_history = {'train': [], 'val': [], 'iter': []}

    for iter_num in range(config.max_iters + 1):
        # Update Learning Rate
        lr = get_lr(iter_num)
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr

        # Evaluate
        if iter_num % config.eval_interval == 0:
            losses = estimate_loss()
            print(f"Step {iter_num:4d} | Train Loss: {losses['train']:.4f} | Val Loss: {losses['val']:.4f} | LR: {lr:.4e}")
            loss_history['iter'].append(iter_num)
            loss_history['train'].append(losses['train'])
            loss_history['val'].append(losses['val'])

        # Forward & Backward Pass
        X, Y = get_batch('train')
        with torch.autocast(device_type=device_type, dtype=ptdtype):
            logits, loss = model(X, Y)

        scaler.scale(loss).backward()

        # Gradient Clipping
        if config.grad_clip != 0.0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)

        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        
        if iter_num % config.log_interval == 0:
            print(f"iter {iter_num}: loss {loss.item():.4f}")
            
    train_time = round(time.time() - start_time, 2)
    
    # Save Outputs
    torch.save(model.state_dict(), f"models/{config.run_name}.pt")
    with open(f"output/{config.run_name}_curves.json", 'w') as f:
        json.dump(loss_history, f)
        
    metrics = {
        "final_train_loss": loss_history['train'][-1],
        "final_val_loss": loss_history['val'][-1],
        "training_time_sec": train_time
    }
    
    log_metrics_to_csv(config, metrics)
    print(f"Finished {config.run_name} in {train_time}s. Saved to models/ and output/")
    
    return model, metrics

In [16]:
pos_encoding_options = ['learned', 'rope', 'alibi']
norm_options = ["layernorm", "none"]
head_options = [6, 1]

experiments = []

# baseline: learned + layernorm + 6 heads
experiments.append(ExperimentConfig(run_name="baseline", pos_encoding="learned", 
                                    norm_type="layernorm", n_head=6))

# Iterate through ablations
for pos_enc, norm, heads in itertools.product(pos_encoding_options, norm_options, head_options):
    # Skip baseline repetition
    if pos_enc == "learned" and norm == "layernorm" and heads == 6:
        continue
    run_name = f"ablation_pos-{pos_enc}_norm-{norm}_heads-{heads}"
    experiments.append(ExperimentConfig(
        run_name=run_name,
        pos_encoding=pos_enc,
        norm_type=norm,
        n_head=heads
    ))

# Run all experiments
for cfg in experiments:
    # Run the model
    model, metrics = train_run(cfg)
    
    # Safely Delete model from memory to avoid Out of Memory (OOM) Errors
    del model
    
    gc.collect()
    
    # Clear PyTorch CUDA cache
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nALL EXPERIMENTS COMPLETED SUCCESSFULLY! Check output/metrics.csv")

Queueing 8 experiments

Starting Run: baseline
Step    0 | Train Loss: 4.2873 | Val Loss: 4.2822
iter 0: loss 4.2630
iter 10: loss 3.4038
iter 20: loss 3.3353
iter 30: loss 3.3662
iter 40: loss 3.2777
iter 50: loss 3.2340
Finished baseline in 838.47s. Saved to models/ and output/metrics.csv

Starting Run: ablation_pos-learned_norm-layernorm_heads-1
Step    0 | Train Loss: 4.2878 | Val Loss: 4.2828
iter 0: loss 4.2631
iter 10: loss 3.4030
iter 20: loss 3.3333
iter 30: loss 3.3577
iter 40: loss 3.2513
iter 50: loss 3.2020
Finished ablation_pos-learned_norm-layernorm_heads-1 in 716.5s. Saved to models/ and output/metrics.csv

Starting Run: ablation_pos-learned_norm-none_heads-6
Step    0 | Train Loss: 4.1742 | Val Loss: 4.1738
iter 0: loss 4.1741
iter 10: loss 3.9876
iter 20: loss 3.4023
iter 30: loss 3.3560
iter 40: loss 3.2342
iter 50: loss 3.1421
Finished ablation_pos-learned_norm-none_heads-6 in 825.23s. Saved to models/ and output/metrics.csv

Starting Run: ablation_pos-learned_norm-

In [22]:
#IF THE CODE IS RUNNING SLOW
#ITS BECAUSE STUFF IS STILL IN VRAM
#run below to clear
#check vram in task manager
# Delete unused variables/models/tensors first
'''
del model
del optimizer
del loss
'''
# Run Python garbage collector
gc.collect()

# Clear PyTorch CUDA cache
torch.cuda.empty_cache()
